# Fraud Detection Under a 100 Alert Review Budget

**Question:** How much fraud can simple models identify when only100 transactions can be reviewed, and what happens when alert rates drift?

Kerem Ozcan | IBM AI Engineering portfolio | Prepared with AI assistance

Original experiment code developed from course concepts. This notebook contains no IBM assessment solutions or supplied course model predictions.

## Design and evaluation boundary

Remove repeated events and conflicting labels, then use chronological60/20/20 cuts without changing the rare-event prevalence. Exclude Time from model inputs; fit the scaler/logistic regression and a depth6 tree exclusively on training records. Include a prevalence baseline. Set a fixed cutoff using validation scores and a100-review budget; ties cannot exceed that validation budget. On the later test period, report both the frozen-cutoff workload and a separate fixed top100 ranking. Test workload can differ under the frozen cutoff. Report average precision (the non-interpolated PR summary), ROC-AUC, precision/recall and actual false alerts. Unit costs are explicitly illustrative.

A two-day historical anonymized dataset. Customer grouping and upstream PCA fitting provenance are unknown. Review costs are illustrative units, not measured production savings. The fixed-cutoff policy cannot guarantee a future alert budget under distribution drift.

## Runtime

Kaggle internet is required for provider-hosted inputs/models. Local verification uses already installed isolated environments. `SECOND_BATCH_FULL=0` selects a small local correctness pilot for the neural projects; the published Kaggle version uses the documented full experiment budget. Data and weights stay in scratch storage; only JSON evidence, plots and notebook outputs are exported.

In [ ]:
import os
import sys
import subprocess
import gzip
import shutil
from pathlib import Path
os.environ["USE_TF"]="0"
os.environ["TOKENIZERS_PARALLELISM"]="false"
os.environ["OMP_NUM_THREADS"]="1"
ON_KAGGLE=Path("/kaggle/working").is_dir()
if ON_KAGGLE and False:
    subprocess.run([sys.executable,"-m","pip","install","-q",*[]],check=True)
import matplotlib.pyplot as plt
import numpy as np
import json
FULL_RUN=os.environ.get("SECOND_BATCH_FULL","1")=="1"
LOCAL_ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/"provenance/source-manifest.json").exists()),Path.cwd())
WORK=Path("/kaggle/working") if ON_KAGGLE else LOCAL_ROOT/".lab-cache/second-batch-notebooks"/'fraud'/"exports"
SCRATCH=Path("/kaggle/temp/ibm-second-batch") if ON_KAGGLE else LOCAL_ROOT/".lab-cache/second-batch-notebooks"/'fraud'/"scratch"
WORK.mkdir(parents=True,exist_ok=True);SCRATCH.mkdir(parents=True,exist_ok=True)
DATA_ROOT=SCRATCH/"inputs" if ON_KAGGLE else LOCAL_ROOT/".lab-cache/second-batch-data"
RUN_OUTPUT=SCRATCH/'fraud';OUTPUT=WORK/'fraud';OUTPUT.mkdir(parents=True,exist_ok=True)


## Inputs and provenance

The [ULB/Worldline credit-card fraud dataset](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud), fetched through the IBM Skills Network teaching mirror. Original dataset conditions remain applicable. No transaction dataset is republished as a notebook output.

Provider downloads must match the pinned SHA-256 values below. Model weights are fetched from the original provider; no submitted lab checkpoint is reused.

In [ ]:
"""Small, standalone data acquisition helpers embedded in the Kaggle releases."""

import hashlib
import shutil
import tarfile
import urllib.request
from pathlib import Path


def verified_download(url: str, target: Path, expected_sha256: str) -> Path:
    """Download to a temporary file and accept only the reviewed dataset bytes."""
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.is_file() and hashlib.sha256(target.read_bytes()).hexdigest() == expected_sha256:
        return target
    temporary = target.with_suffix(target.suffix + ".download")
    try:
        with urllib.request.urlopen(url, timeout=60) as response, temporary.open("wb") as stream:
            shutil.copyfileobj(response, stream)
        if hashlib.sha256(temporary.read_bytes()).hexdigest() != expected_sha256:
            raise ValueError("Downloaded data changed; review its provenance before running.")
        temporary.replace(target)
    finally:
        temporary.unlink(missing_ok=True)
    return target


def unpack_regular_tar(archive: Path, destination: Path) -> None:
    """Reject links, special files and traversal before writing any archive member."""
    root = destination.resolve()
    with tarfile.open(archive) as bundle:
        members = bundle.getmembers()
        for member in members:
            path = root / member.name
            if (
                Path(member.name).is_absolute()
                or not path.resolve().is_relative_to(root)
                or not (member.isfile() or member.isdir())
            ):
                raise ValueError("Unsafe dataset archive member.")
        for member in members:
            path = root / member.name
            if member.isdir():
                path.mkdir(parents=True, exist_ok=True)
            else:
                path.parent.mkdir(parents=True, exist_ok=True)
                with bundle.extractfile(member) as source, path.open("wb") as target:
                    shutil.copyfileobj(source, target)


In [ ]:
INPUTS=[{'file': 'fraud/creditcard.csv', 'url': 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-ML0101EN-SkillsNetwork/labs/Module%203/data/creditcard.csv', 'sha256': '76274b691b16a6c49d3f159c883398e03ccd6d1ee12d9d8ee38f4b4b98551a89', 'bytes': 150828752}]
for entry in INPUTS:
    path=verified_download(entry["url"],DATA_ROOT/entry["file"],entry["sha256"])
    if path.suffix==".gz":
        with gzip.open(path,"rb") as source,path.with_suffix("").open("wb") as target:
            shutil.copyfileobj(source,target)


## Standalone implementation

Expand this cell to inspect data partitions, actual sample accounting, training and selected-checkpoint reload checks.

In [ ]:
"""Shared provenance and evaluation helpers for the selected portfolio projects."""

from __future__ import annotations

import hashlib
import importlib.metadata
import json
import platform
import random
from collections import Counter, defaultdict
from datetime import UTC, datetime
from pathlib import Path


def file_sha256(path: Path) -> str:
    """Hash an input/checkpoint without reading the entire file into memory."""
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def write_json(path: Path, value: dict | list) -> None:
    """Publish a complete JSON artifact atomically."""
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False) + "\n", encoding="utf-8"
    )
    temporary.replace(path)


def runtime(packages: list[str]) -> dict:
    """Capture versions without collecting environment variables or host paths."""
    return {
        "utc": datetime.now(UTC).isoformat(),
        "python": platform.python_version(),
        "platform": platform.system(),
        "packages": {name: importlib.metadata.version(name) for name in packages},
    }


def source_identity(project_file: str | None, embedded_digest: str | None = None) -> dict:
    """Bind module runs and generated notebooks to their authored source files."""
    project_digest = file_sha256(Path(project_file)) if project_file else embedded_digest
    common_digest = (
        file_sha256(Path(__file__))
        if "__file__" in globals()
        else globals().get("COMMON_SOURCE_SHA256")
    )
    if not project_digest or not common_digest:
        raise ValueError("Project source identity is missing.")
    return {"project_sha256": project_digest, "common_sha256": common_digest}


def index_images(
    root: Path, class_names: list[str], directories: dict[str, str] | None = None
) -> list[dict]:
    """Index decoded RGB content as well as bytes, retaining only relative filenames."""
    from PIL import Image, ImageOps

    rows = []
    directories = directories or {"unsplit": "."}
    for role, directory in directories.items():
        for label, name in enumerate(class_names):
            folder = root / directory / name
            if not folder.is_dir():
                raise ValueError(f"Missing image class directory: {directory}/{name}")
            for path in sorted(folder.rglob("*")):
                if path.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
                    continue
                with Image.open(path) as original:
                    image = ImageOps.exif_transpose(original).convert("RGB")
                    pixels = str(image.size).encode() + image.tobytes()
                rows.append(
                    {
                        "file": path.relative_to(root).as_posix(),
                        "label": label,
                        "role": role,
                        "file_sha256": file_sha256(path),
                        "pixel_sha256": hashlib.sha256(pixels).hexdigest(),
                    }
                )
    if not rows:
        raise ValueError("No images found.")
    return rows


def clean_image_splits(rows: list[dict]) -> tuple[list[dict], list[dict]]:
    """Remove exact duplicates and conflicting labels, preserving held-out copies first."""
    groups = defaultdict(list)
    for row in rows:
        groups[row["pixel_sha256"]].append(row)
    retained, removed = [], []
    priority = {"test": 0, "validation": 1, "train": 2}
    for group in groups.values():
        if len({row["label"] for row in group}) > 1:
            removed.extend(
                {"file": row["file"], "reason": "conflicting_pixel_labels"} for row in group
            )
            continue
        ordered = sorted(group, key=lambda row: (priority[row["role"]], row["file"]))
        retained.append(ordered[0])
        removed.extend(
            {"file": row["file"], "reason": "exact_pixel_duplicate"} for row in ordered[1:]
        )
    return sorted(retained, key=lambda row: row["file"]), removed


def balanced_subset(rows: list[dict], per_class: int | None, seed: int) -> list[dict]:
    """Select a reproducible pilot without changing the complete split manifest."""
    if per_class is None:
        return rows
    rng = random.Random(seed)
    result = []
    for label in sorted({row["label"] for row in rows}):
        group = sorted((row for row in rows if row["label"] == label), key=lambda row: row["file"])
        rng.shuffle(group)
        result.extend(group[:per_class])
    return sorted(result, key=lambda row: row["file"])


def split_counts(rows: list[dict]) -> dict:
    """Describe the actual records, including class support."""
    return {
        role: dict(Counter(str(row["label"]) for row in rows if row["role"] == role))
        for role in ["train", "validation", "test"]
    }


def classification_metrics(labels: list[int], probabilities: list[float]) -> dict:
    """Evaluate exactly the predictions supplied, never the size of an upstream dataset."""
    import numpy as np
    from sklearn.metrics import (
        accuracy_score,
        classification_report,
        confusion_matrix,
        f1_score,
        roc_auc_score,
    )

    if not labels or len(labels) != len(probabilities):
        raise ValueError("Predictions and labels must have the same nonzero length.")
    probabilities = np.asarray(probabilities)
    if not np.isfinite(probabilities).all() or ((probabilities < 0) | (probabilities > 1)).any():
        raise ValueError("Expected finite probabilities in [0, 1].")
    predicted = (probabilities >= 0.5).astype(int)
    return {
        "samples": len(labels),
        "accuracy": float(accuracy_score(labels, predicted)),
        "macro_f1": float(
            f1_score(labels, predicted, labels=[0, 1], average="macro", zero_division=0)
        ),
        "roc_auc": float(roc_auc_score(labels, probabilities)) if len(set(labels)) == 2 else None,
        "confusion_matrix": confusion_matrix(labels, predicted, labels=[0, 1]).tolist(),
        "per_class": classification_report(
            labels, predicted, labels=[0, 1], output_dict=True, zero_division=0
        ),
    }

COMMON_SOURCE_SHA256='ded7059d91e2123ef7bc9be2823a1d220d905041f45b9dbe83bac59d27547853'
PROJECT_SOURCE_SHA256='dd496d6e14c7769a9cce2aef7737be13615b7edaf6847e394bf84c751f1aea07'
"""Evaluate train-only preprocessing and validation thresholds on later transactions."""


import time
from pathlib import Path

import numpy as np
import pandas as pd



def chronological_splits(frame: pd.DataFrame) -> tuple[dict, dict]:
    """Remove repeated events and split at time boundaries, preserving natural prevalence."""
    required = {"Time", "Amount", "Class", *(f"V{i}" for i in range(1, 29))}
    if not required.issubset(frame.columns) or not set(frame["Class"]).issubset({0, 1}):
        raise ValueError("Expected the credit-card dataset's numeric features and binary labels.")
    if not np.isfinite(frame[list(required)].to_numpy()).all():
        raise ValueError("Missing or non-finite transaction values.")
    data = frame.copy()
    data["record_id"] = np.arange(len(data))
    event_columns = sorted(required - {"Class"})
    conflicting = data.groupby(event_columns, dropna=False)["Class"].transform("nunique") > 1
    conflicts = data.loc[conflicting, "record_id"].tolist()
    data = data.loc[~conflicting]
    duplicates = data.duplicated(event_columns)
    repeated = data.loc[duplicates, "record_id"].tolist()
    data = data.loc[~duplicates].sort_values(["Time", "record_id"], kind="stable")
    cut1, cut2 = data["Time"].quantile([0.6, 0.8]).tolist()
    parts = {
        "train": data[data.Time < cut1],
        "validation": data[(data.Time >= cut1) & (data.Time < cut2)],
        "test": data[data.Time >= cut2],
    }
    if any(set(rows.Class) != {0, 1} for rows in parts.values()):
        raise ValueError("Every chronological partition must contain both labels.")
    audit = {
        "conflicting_ids": conflicts,
        "repeated_event_ids": repeated,
        "time_cuts": [cut1, cut2],
    }
    return parts, audit


def threshold_for_budget(scores, budget: int) -> float:
    """Choose a validation-only cutoff; ties must never exceed its review budget."""
    scores = np.asarray(scores, dtype=float)
    if budget < 1 or not len(scores) or not np.isfinite(scores).all():
        raise ValueError("A positive budget and finite nonempty scores are required.")
    if budget >= len(scores):
        return float(scores.min())
    boundary = float(np.sort(scores)[-budget])
    return (
        float(np.nextafter(boundary, np.inf)) if (scores >= boundary).sum() > budget else boundary
    )


def alert_metrics(labels, predicted) -> dict:
    """Describe actual alerts and an explicitly illustrative unit-cost scenario."""
    labels, predicted = np.asarray(labels), np.asarray(predicted, dtype=bool)
    tp = int(((labels == 1) & predicted).sum())
    fp = int(((labels == 0) & predicted).sum())
    fn = int(((labels == 1) & ~predicted).sum())
    return {
        "alerts": int(predicted.sum()),
        "true_alerts": tp,
        "false_alerts": fp,
        "missed_fraud": fn,
        "precision": tp / max(1, tp + fp),
        "recall": tp / max(1, tp + fn),
        "illustrative_cost": int(predicted.sum()) + 100 * fn,
    }


def run(data_file: Path, output: Path, *, seed: int = 42, budget: int = 100) -> dict:
    """Train fresh baselines, freeze validation cutoffs and evaluate the complete later test set."""
    import joblib
    from sklearn.dummy import DummyClassifier
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import average_precision_score, precision_recall_curve, roc_auc_score
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.tree import DecisionTreeClassifier

    started = time.perf_counter()
    parts, exclusions = chronological_splits(pd.read_csv(data_file))
    features = [f"V{i}" for i in range(1, 29)] + ["Amount"]
    manifest = {role: rows.record_id.astype(int).tolist() for role, rows in parts.items()}
    write_json(output / "split-manifest.json", {"partitions": manifest, "exclusions": exclusions})
    train, validation, test = [parts[role] for role in ["train", "validation", "test"]]
    factories = {
        "prevalence_baseline": DummyClassifier(strategy="prior"),
        "logistic_regression": make_pipeline(
            StandardScaler(),
            LogisticRegression(class_weight="balanced", max_iter=1000, random_state=seed),
        ),
        "decision_tree": DecisionTreeClassifier(
            max_depth=6, min_samples_leaf=20, class_weight="balanced", random_state=seed
        ),
    }
    results = {}
    for name, model in factories.items():
        model.fit(train[features], train.Class)
        validation_scores = model.predict_proba(validation[features])[:, 1]
        cutoff = threshold_for_budget(validation_scores, budget)
        scores = model.predict_proba(test[features])[:, 1]
        selected = scores >= cutoff
        ranked = np.zeros(len(test), dtype=bool)
        ranked[np.argsort(-scores, kind="stable")[: min(budget, len(test))]] = True
        precision, recall, _ = precision_recall_curve(test.Class, scores)
        keep = np.unique(np.linspace(0, len(precision) - 1, min(256, len(precision)), dtype=int))
        path = output / f"{name}.joblib"
        joblib.dump(model, path)
        reloaded = joblib.load(path)  # Only the model created directly above is deserialized.
        np.testing.assert_array_equal(
            model.predict_proba(test[features].iloc[:16]),
            reloaded.predict_proba(test[features].iloc[:16]),
        )
        results[name] = {
            "average_precision": float(average_precision_score(test.Class, scores)),
            "roc_auc": float(roc_auc_score(test.Class, scores)),
            "validation_cutoff": cutoff,
            "validation_alerts": int((validation_scores >= cutoff).sum()),
            "fixed_threshold_test": alert_metrics(test.Class, selected),
            "top_budget_test": alert_metrics(test.Class, ranked),
            "test_samples": len(test),
            "exact_reload": True,
            "checkpoint_sha256": file_sha256(path),
            "pr_curve": {"precision": precision[keep].tolist(), "recall": recall[keep].tolist()},
        }
    report = {
        "project": "fraud",
        "profile": "chronological-complete-splits",
        "seed": seed,
        "data_sha256": file_sha256(data_file),
        "split_manifest_sha256": file_sha256(output / "split-manifest.json"),
        "split_counts": {
            role: {
                "samples": len(rows),
                "fraud": int(rows.Class.sum()),
                "min_time": float(rows.Time.min()),
                "max_time": float(rows.Time.max()),
            }
            for role, rows in parts.items()
        },
        "review_budget": budget,
        "features": features,
        "models": results,
        "threshold_policy": f"Validation-only fixed cutoff; future alert counts can drift. Separately report a fixed top-{budget} batch ranking.",
        "cost_assumptions": "Illustration only: one unit per reviewed alert and 100 units per missed fraud; no production cost estimate.",
        "limitations": "Two-day historical anonymized dataset; unknown customer grouping and upstream PCA fitting provenance. No deployment or realized financial benefit claimed.",
        "seconds": time.perf_counter() - started,
        "runtime": runtime(["numpy", "pandas", "scikit-learn", "joblib"]),
        "source": source_identity(
            globals().get("__file__"), globals().get("PROJECT_SOURCE_SHA256")
        ),
    }
    write_json(output / "metrics.json", report)
    return report

In [ ]:
report=run(DATA_ROOT/'fraud/creditcard.csv', RUN_OUTPUT)
for artifact in RUN_OUTPUT.glob("*.json"):
    shutil.copyfile(artifact,OUTPUT/artifact.name)
print("Executed profile:",report["profile"])
print("Actual runtime:",report["runtime"])
print("Source identity:",report["source"])


## Freshly measured results

These values are generated by this execution. Keep unsuccessful baselines and the stated evaluation limits visible.

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(11,4))
for name,result in report["models"].items():
    curve=result["pr_curve"]
    axes[0].plot(curve["recall"],curve["precision"],label=f"{name} AP={result['average_precision']:.3f}")
names=list(report["models"])
axes[1].bar(names,[report["models"][n]["top_budget_test"]["true_alerts"] for n in names],color="#138477")
axes[0].set(xlabel="Recall",ylabel="Precision",title="Later-period test precision/recall");axes[0].legend(fontsize=8)
axes[1].set(ylabel="Fraud found in top100 reviews",title="Fixed batch review budget");axes[1].tick_params(axis="x",labelrotation=25)
fig.tight_layout();fig.savefig(OUTPUT/"results.png",dpi=160);plt.show()
print("Actual split counts:",report["split_counts"])
for name,result in report["models"].items():print(name,result["fixed_threshold_test"],result["top_budget_test"])
print(report["cost_assumptions"])


## Interpretation and attribution

A two-day historical anonymized dataset. Customer grouping and upstream PCA fitting provenance are unknown. Review costs are illustrative units, not measured production savings. The fixed-cutoff policy cannot guarantee a future alert budget under distribution drift.

The [ULB/Worldline credit-card fraud dataset](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud), fetched through the IBM Skills Network teaching mirror. Original dataset conditions remain applicable. No transaction dataset is republished as a notebook output.

Concepts studied through the [IBM AI Engineering Professional Certificate](https://www.coursera.org/professional-certificates/ai-engineer). Original provider conditions remain applicable. Mechanical execution and reload checks do not replace task-specific or human quality evaluation.